In [ ]:
import sympy as sp
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output, Math

# ==============================================================================
# PROBLEM 8: Fully Symbolic Z-Transform via SymPy Definition Sum & Visualization
# Signal: Triangular pulse x[n]
# ==============================================================================

explanation_text = """
<div style="background-color: #f8f9fa; padding: 10px; border-radius: 5px; border: 1px solid #dee2e6; font-size: 13px;">
<b>Solution Overview (Fully Symbolic Calculation via SymPy)</b><br>
* <b>Signal:</b> Triangular pulse defined piecewise for 0 ≤ n ≤ 2N.<br>
* <b>Z-Transform Definition:</b> X(z) = Σ x[n] z<sup>-n</sup> computed explicitly via SymPy summation.<br>
* <b>ROC (Region of Convergence):</b> Entire z-plane except z = 0 (finite duration signal).<br>
* <b>Note:</b> Use the slider below to dynamically change parameter N and observe the exact symbolic expansion and plots.
</div>
"""
display(widgets.HTML(explanation_text))

out = widgets.Output()

# Ορισμός συμβολικών μεταβλητών
n_sym = sp.Symbol('n', integer=True)
z_sym = sp.Symbol('z', complex=True)
N_sym = sp.Symbol('N', integer=True, positive=True)

# 1. Συμβολικός υπολογισμός Z-Transform χωρισμένος στα κομμάτια του σήματος
S1_sym = sp.summation(n_sym * z_sym**(-n_sym), (n_sym, 0, N_sym))
S2_sym = sp.summation((2*N_sym - n_sym) * z_sym**(-n_sym), (n_sym, N_sym + 1, 2*N_sym))
X_z_sym = sp.simplify(S1_sym + S2_sym)

display(Math(f"X(z) = {sp.latex(X_z_sym)}"))

def plot_problem_8(N_val):
    with out:
        clear_output(wait=True)
        
        fig, (ax_pz, ax_time) = plt.subplots(1, 2, figsize=(16, 5), gridspec_kw={'width_ratios': [1, 2]})
        plt.subplots_adjust(wspace=0.25)

        # --- 1. Pole-Zero Map & ROC ---
        ax_pz.set_aspect('equal')
        ax_pz.set_xlim(-2.0, 2.0)
        ax_pz.set_ylim(-2.0, 2.0)
        ax_pz.axhline(0, color='black', linewidth=1)
        ax_pz.axvline(0, color='black', linewidth=1)
        ax_pz.grid(True, linestyle=':', alpha=0.7)

        # ROC: Finite duration signal -> ROC is entire z-plane except z = 0
        x_vals = np.linspace(-2.5, 2.5, 400)
        y_vals = np.linspace(-2.5, 2.5, 400)
        X, Y = np.meshgrid(x_vals, y_vals)
        Z_dist = np.sqrt(X**2 + Y**2)
        roc_mask = Z_dist > 0.05

        ax_pz.imshow(roc_mask, extent=(-2.5, 2.5, -2.5, 2.5), origin='lower', cmap='Greens', alpha=0.25, zorder=0)

        theta = np.linspace(0, 2*np.pi, 200)
        
        # Σχεδίαση μοναδιαίου κύκλου ως ορίου/αναφοράς στην περιοχή σύγκλισης
        ax_pz.plot(np.cos(theta), np.sin(theta), 'k--', alpha=0.5)
        
        # Υπολογισμός μηδενικών
        if N_val > 0:
            k_vals = np.arange(1, N_val)
            zeros_angle = 2 * np.pi * k_vals / N_val
            zeros_x = np.cos(zeros_angle)
            zeros_y = np.sin(zeros_angle)
            ax_pz.scatter(zeros_x, zeros_y, s=100, facecolors='none', edgecolors='b', linewidths=2, marker='o')

        # Πόλοι στη θέση z = 0
        ax_pz.scatter([0], [0], s=140, color='purple', marker='x', linewidths=3)

        ax_pz.set_title(f'Pole-Zero Map & ROC (N = {N_val})', fontsize=10, fontweight='bold')
        ax_pz.set_xlabel('Real Part', fontsize=9)
        ax_pz.set_ylabel('Imaginary Part', fontsize=9)

        # Υπόμνημα σε μία ευθεία (3 αντικείμενα, ncol=3)
        unit_circle_handle = plt.Line2D([0], [0], color='k', linestyle='--', alpha=0.5, label='Unit Circle')
        pole_handle = plt.Line2D([0], [0], marker='x', color='purple', markersize=8, markeredgewidth=3, linestyle='None', label=f'Multiple Pole at z=0 (order {2*N_val-1})')
        zero_handle = plt.Line2D([0], [0], marker='o', markerfacecolor='none', markeredgecolor='b', markersize=8, markeredgewidth=2, linestyle='None', label='Zeros')
        
        ax_pz.legend(handles=[unit_circle_handle, pole_handle, zero_handle], loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=3, fontsize=8)

        # --- 2. Time Domain Plot ---
        n_vec = np.arange(0, 2 * N_val + 1)
        x_n_vals = np.piecewise(n_vec, 
                                [n_vec <= N_val, n_vec > N_val], 
                                [lambda n: n, lambda n: 2 * N_val - n])

        ax_time.stem(n_vec, x_n_vals, linefmt='r-', markerfmt='ro', basefmt='k-')
        ax_time.set_title(f'Temporal Evolution: Triangular Pulse (N = {N_val})', fontsize=10, fontweight='bold')
        ax_time.set_xlabel('Time index n', fontsize=9)
        ax_time.set_ylabel('x[n]', fontsize=9)
        ax_time.set_xlim(-1, 2 * N_val + 2)
        ax_time.set_ylim(-0.5, N_val + 1)
        ax_time.grid(True, linestyle=':', alpha=0.7)

        plt.show()

# Slider για το N
N_slider = widgets.IntSlider(value=4, min=2, max=10, step=1, description='N:', style={'description_width': 'initial'})

plot_problem_8(N_slider.value)

interactive_plot = widgets.interactive(plot_problem_8, N_val=N_slider)
display(widgets.VBox([interactive_plot, out]))